# 02 — Interpolation: what you interpolate matters more than how

A yield curve is a set of nodes $(t_i, P(0,t_i))$ plus a rule for everything in between. The rule
determines the *instantaneous forward* $f(t) = -\partial_t \log P(0,t)$, and forwards are what a
trader sees: a 3m forward strip, a curve's "shape". Interpolating the wrong quantity produces
forwards that oscillate, go negative, or jump — and a Bermudan priced off such a curve inherits every
artefact.

Write $y(t) = -\log P(0,t) = r(t)\,t$. Then $f(t) = y'(t)$ and the *discrete forward* over a node
interval is the average of $f$:
$$
f^d_i = \frac{y_i - y_{i-1}}{t_i - t_{i-1}} .
$$

## 1. Linear on $y$ (= log-linear on discount factors)

$y$ piecewise linear ⇒ $f$ piecewise **constant** at $f^d_i$: never negative if $y$ is increasing,
local (a bumped node moves two intervals), but the forward curve is a staircase. Market default for OIS
curves because it is local and boring, which is what risk wants.

## 2. Monotone convex (Hagan & West 2006)

Interpolate $f$ itself with a quadratic on each interval, constrained so that its integral equals
$f^d_i$, it is continuous at nodes, and it stays between $0$ and $2\min(f^d_i, f^d_{i+1})$. Node
forwards use the "central" estimate

$$
f_i = \frac{(t_i - t_{i-1})\, f^d_{i+1} + (t_{i+1} - t_i)\, f^d_i}{t_{i+1} - t_{i-1}}, \qquad
f_0 = f^d_1 - \tfrac12 (f_1 - f^d_1), \qquad f_n = f^d_n - \tfrac12(f_{n-1} - f^d_n),
$$

and on each interval, with $x = (t - t_{i-1})/(t_i - t_{i-1})$, $g_0 = f_{i-1} - f^d_i$, $g_1 = f_i - f^d_i$,
the simplest regime is
$$
f(t) = f^d_i + g_0\,(1 - 4x + 3x^2) + g_1\,(-2x + 3x^2),
$$
which integrates to $f^d_i$ over the interval for any $g_0, g_1$. The three other regimes replace the
quadratic by a flat-then-quadratic piece so that $f$ cannot overshoot when $g_0$ and $g_1$ have
awkward signs — that is the "monotone" part.

### Worked example by hand

Nodes $t = 0, 1, 2, 3$ with zero rates $3\%, 4\%, 4.5\%$: $y = 0, 0.03, 0.08, 0.135$.
Discrete forwards: $f^d_1 = 0.03$, $f^d_2 = 0.05$, $f^d_3 = 0.055$.
Node forward at $t = 1$: $f_1 = (1\cdot 0.05 + 1\cdot 0.03)/2 = 0.04$; at $t = 2$: $f_2 = (0.055 + 0.05)/2 = 0.0525$.
End: $f_0 = 0.03 - \tfrac12(0.04 - 0.03) = 0.025$; $f_3 = 0.055 - \tfrac12(0.0525 - 0.055) = 0.05625$.
On the middle interval $g_0 = 0.04 - 0.05 = -0.01$, $g_1 = 0.0525 - 0.05 = 0.0025$: signs differ and
$|g_1| < \tfrac12 |g_0|$, so this is regime (iii) of the paper — the forward is flat near $t=2$ and
bends only near $t=1$. Verify against `mc_f` below at $t = 1.5$ once the code is written.

## 3. Natural cubic spline on $y$

Smooth ($C^2$) and pleasant to look at, but every node influences every interval, and the second
derivative is forced to zero at the ends. Forwards can go negative between nodes on a steep curve, and
a bump at 10y moves the 2y forward. Fine for a vol surface; wrong for a discount curve.

In [ ]:
import _util
from _util import run_demo
import matplotlib.pyplot as plt

nodes = ["0:0", "0.5:0.020", "1:0.024", "2:0.030", "3:0.034", "5:0.039", "7:0.042", "10:0.045", "15:0.048", "20:0.050", "30:0.052"]
df = run_demo("rl_demo_interp", 600, *nodes)
df.head()

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for col, lab in [("linear_y", "linear on y"), ("mc_y", "monotone convex"), ("cubic_y", "natural cubic")]:
    a1.plot(df["t"], df[col] / df["t"].where(df["t"] > 0), label=lab)
a1.set_ylabel("zero rate y(t)/t"); a1.legend(); a1.set_title("Zero rates look identical...")
for col, lab in [("linear_f", "linear on y (flat forwards)"), ("mc_f", "monotone convex"), ("cubic_f", "natural cubic")]:
    a2.plot(df["t"], df[col], label=lab)
a2.axhline(0, color="k", lw=0.5)
a2.set_ylabel("instantaneous forward f(t)"); a2.set_xlabel("t (years)"); a2.legend()
a2.set_title("...the forwards do not: staircase vs continuous vs overshooting")
plt.tight_layout(); plt.show()

In [ ]:
# Hand example check: t = 0,1,2,3 with zero rates 3%, 4%, 4.5%.
small = run_demo("rl_demo_interp", 300, "0:0", "1:0.03", "2:0.04", "3:0.045")
for t, expected in [(0.0, 0.025), (1.0, 0.04), (2.0, 0.0525), (3.0, 0.05625)]:
    row = small.iloc[(small["t"] - t).abs().argmin()]
    print(f"t={t}: mc_f={row['mc_f']:.6f}  hand={expected:.6f}")

In [ ]:
# Locality: bump the 10y node by 10bp and see which forwards move under each scheme.
bumped = [n if not n.startswith("10:") else "10:0.046" for n in nodes]
db = run_demo("rl_demo_interp", 600, *bumped)
fig, ax = plt.subplots(figsize=(9, 3.5))
for col, lab in [("linear_f", "linear"), ("mc_f", "monotone convex"), ("cubic_f", "natural cubic")]:
    ax.plot(df["t"], (db[col] - df[col]) * 1e4, label=lab)
ax.set_xlabel("t"); ax.set_ylabel("Δ forward (bp)"); ax.legend(); ax.set_title("10y node +10bp: where does the forward move?")
plt.show()